# 08 — Context Engineering

## Scenario
Northstar has a bot that automatically approves or denies user requests based on the internal policy.

**The Danger:** If we just dump data into the prompt without structure or boundaries, a malicious user can "poison" the context by hiding instructions inside their data. This is known as **Prompt Injection**.


In [ ]:
from pathlib import Path
import sys

COURSE_DIR = Path.cwd()
sys.path.insert(0, str(COURSE_DIR))
from northstar.runtime import get_client
from lab08 import CASES, USER_DATA, build_packet, build_requests, decide, policy_allows, requested_nodes
from northstar.security import instruction_like_score


def show_request(request):
    print("SYSTEM:\n", request.system)
    for message in request.messages:
        if message.text:
            print(f"{message.role.upper()}:\n{message.text}")
        for part in message.parts:
            print(f"{message.role.upper()} PART:", part)

client = get_client(COURSE_DIR / "fixtures/replays.json")


## Step 1: The "Data Dump" (Vulnerable Baseline)

Watch what happens when we just smash the policy and the user data together in a single string.


In [ ]:
request = next(r for r in build_requests() if r.case_id == "i08/naive/injection-20-nodes")
show_request(request)
response = client.generate(request)
print("RECORDED RESPONSE:\n", response.text)
print("PARSED ANSWER:", response.text)
assert response.text == "YES"
assert instruction_like_score(USER_DATA) >= 0.8


## Step 2: Context Engineering with Delimiters

We must teach the model the difference between "our instructions" and "untrusted user data". 
The industry standard is to use **XML tags** to create strict boundaries within the context window.


In [ ]:
request = next(r for r in build_requests() if r.case_id == "i08/engineered/compliant-3-nodes")
show_request(request)
response = client.generate(request)
print("RECORDED RESPONSE:\n", response.text)
print("PARSED ANSWER:", response.text)
requested = requested_nodes(CASES[1]["user_data"])
assert decide(response.text, requested) == "approved"
assert policy_allows(requested)


## Step 3: Budget and application control

Context selection is also an application responsibility: preserve high-priority policy while pruning low-priority history.


In [ ]:
sections = [("policy", "At most 5 nodes may be requested.", 100), ("customer request", USER_DATA, 50), ("chat history", "Old conversation", 1)]
packet = build_packet(sections, budget_tokens=15)
print(packet)
assert "[policy]" in packet
assert "[chat history]" not in packet
request = next(r for r in build_requests() if r.case_id == "i08/engineered/polite-7-nodes")
show_request(request)
response = client.generate(request)
print("RECORDED RESPONSE:", response.text)
print("PARSED DECISION:", decide(response.text, requested_nodes(CASES[2]["user_data"])))
assert decide(response.text, requested_nodes(CASES[2]["user_data"])) == "rejected"


## Takeaway
The recorded context-control run scored the injection at 0.8, approved the compliant 3-node request, rejected the 7-node request, and retained policy while dropping chat history.


## References
- [Core Concepts & Workflow](README.md#core-concepts--workflow)
- [Deep dive](README.md#deep-dive)
- [Production Best Practices](README.md#production-best-practices)
